# 🌊 Flood Digital Twin: From Data to Decision Support
### Workshop Practice Notebook · 60 min · Mixed Levels

---

This notebook guides you through building a **simplified Digital Twin (DT)** for flood disaster management.  
You will follow the same conceptual pipeline used in real-world urban DT systems.

| Step | Topic | DT Periodic Table Element |
|------|-------|:---:|
| 1 | The Framework — synthetic city layers | *Foundation* |
| 2 | Real precipitation data via open API | **Da** — Data Acquisition |
| 3 | Flood susceptibility simulation | **Si** — Simulation |
| 4 | Interactive geospatial map | **Vi** — Visualization |
| 5 | Risk scoring + decision recommendations | **An** — Analytics |

---
> 🔰 **Beginner:** Read every cell and run them top-to-bottom. No code changes needed.  
> 🚀 **Advanced:** Look for `# 🚀 EXTENSION` comments — optional deeper challenges throughout.  
> ⚠️ **Run all cells in order.** Each cell builds on the previous one.


---
## ⚙️ Cell 1 — Setup
Run this cell first. It installs two lightweight packages and imports everything needed.


In [ ]:
# ── INSTALL (run once, ~30 seconds) ──────────────────────────────────────────
!pip install folium requests --quiet

# ── IMPORTS ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
import requests
import folium
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

print("✅ Setup complete!")
print(f"   numpy {np.__version__}  |  pandas {pd.__version__}  |  folium {folium.__version__}")


---
## Part 1 — Digital Twin Periodic Table

The **Digital Twin Periodic Table** (Digital Twin Consortium, 2020) organises every capability  
of a DT system into structured element groups — similar to how chemistry organises matter.

The three **highlighted elements** below are what this notebook implements:

| Symbol | Element | What we build |
|--------|---------|---------------|
| **Da** | Data Acquisition | Fetch real hourly precipitation from a global API |
| **Si** | Simulation | Run a flood susceptibility model on the city grid |
| **An** | Analytics | Score risk per zone and generate recommendations |

Run the cell below to see the periodic table with our elements highlighted.


In [ ]:
# ── DT PERIODIC TABLE — simplified visual ────────────────────────────────────
# Symbol, Name, highlighted?
ELEMENTS = [
    ("Id",  "Identity",         False, "#adb5bd"),
    ("St",  "State",            False, "#adb5bd"),
    ("Md",  "Metadata",         False, "#adb5bd"),
    ("Re",  "Relationship",     False, "#adb5bd"),
    ("Da",  "Data\nAcquisition",True,  "#4cc9f0"),  # ← WE USE THIS
    ("Sc",  "Sync",             False, "#adb5bd"),
    ("Pr",  "Processing",       False, "#adb5bd"),
    ("Si",  "Simulation",       True,  "#f72585"),  # ← WE USE THIS
    ("Pm",  "Parametric\nModel",False, "#adb5bd"),
    ("Bh",  "Behaviour",        False, "#adb5bd"),
    ("An",  "Analytics",        True,  "#7209b7"),  # ← WE USE THIS
    ("Ml",  "Machine\nLearning",False, "#adb5bd"),
    ("Dp",  "Decision\nProcess",False, "#adb5bd"),
    ("Vi",  "Visualization",    False, "#adb5bd"),
    ("Ui",  "User\nInterface",  False, "#adb5bd"),
    ("Ap",  "API",              False, "#adb5bd"),
]

fig, ax = plt.subplots(figsize=(15, 2.8))
ax.set_xlim(0, len(ELEMENTS))
ax.set_ylim(0, 1)
ax.axis("off")
fig.suptitle("Digital Twin Periodic Table — Elements Used in This Notebook Are Highlighted",
             fontsize=11, fontweight="bold", y=1.04)

for i, (sym, name, highlight, color) in enumerate(ELEMENTS):
    fc = color if highlight else "#f1f3f5"
    ec = "#111" if highlight else "#ced4da"
    lw = 2.8 if highlight else 0.8
    rect = mpatches.FancyBboxPatch(
        (i + 0.06, 0.06), 0.86, 0.88,
        boxstyle="round,pad=0.04", facecolor=fc, edgecolor=ec, linewidth=lw
    )
    ax.add_patch(rect)
    tc = "white" if highlight else "#868e96"
    ax.text(i + 0.49, 0.68, sym,  ha="center", va="center",
            fontsize=10.5, fontweight="bold", color=tc)
    ax.text(i + 0.49, 0.28, name, ha="center", va="center",
            fontsize=5.5, color=tc, linespacing=1.3)

plt.tight_layout()
plt.show()
print("\n  🔵 Da (Data Acquisition)    🔴 Si (Simulation)    🟣 An (Analytics)")
print("  These three elements form the core of this 60-minute practice session.")


---
## Part 2 — The Physical Entity: Synthetic Coastal City

Every Digital Twin starts with a **physical entity** — the real-world system being mirrored.

We use a synthetic **20 × 20 grid** (400 cells, each ~500 m × 500 m) representing a generic  
coastal city. Each cell carries three static layers:

| Layer | Description | Role in DT |
|-------|-------------|-----------|
| 🏔️ **Elevation** | Terrain height (meters) | Determines flood accumulation |
| 🏙️ **Land Use** | Residential / Commercial / Industrial / Green | Controls runoff & infiltration |
| 👥 **Population** | Proxy density (people/cell) | Drives risk exposure scoring |

> 🚀 **Extension:** Replace this synthetic grid with a real DEM from  
> [OpenTopography](https://opentopography.org) and buildings from OpenStreetMap via `osmnx`.


In [ ]:
# ── CITY SELECTION ────────────────────────────────────────────────────────────
# Pick one city. The real precipitation API will fetch data for these coordinates.

CITIES = {
    "Houston, USA":           {"lat":  29.76, "lon": -95.37},
    "Jakarta, Indonesia":     {"lat":  -6.21, "lon": 106.85},
    "Mumbai, India":          {"lat":  19.08, "lon":  72.88},
    "Rotterdam, Netherlands": {"lat":  51.92, "lon":   4.48},
    "Lagos, Nigeria":         {"lat":   6.52, "lon":   3.38},
    "Bangkok, Thailand":      {"lat":  13.75, "lon": 100.52},
    "New Orleans, USA":       {"lat":  29.95, "lon": -90.07},
}

# ← CHANGE THIS LINE to explore a different city
SELECTED = "Houston, USA"

LAT       = CITIES[SELECTED]["lat"]
LON       = CITIES[SELECTED]["lon"]
CITY_NAME = SELECTED
GRID      = 20     # 20×20 grid

print(f"🌍  City selected : {CITY_NAME}")
print(f"    Coordinates  : ({LAT:.2f}°, {LON:.2f}°)")
print(f"    Grid         : {GRID}×{GRID} = {GRID**2} cells  (~{GRID*0.5:.0f} km × {GRID*0.5:.0f} km)")


In [ ]:
# ── BUILD SYNTHETIC TERRAIN ───────────────────────────────────────────────────
np.random.seed(42)
x = np.linspace(0, 1, GRID)
y = np.linspace(0, 1, GRID)
XX, YY = np.meshgrid(x, y)

# Elevation: rises west→east (coast on left), undulates N–S, random noise
elevation = (
    6.0 * XX
    + 0.6 * np.sin(YY * np.pi * 2.5)
    + np.random.normal(0, 0.45, (GRID, GRID))
    + 0.4
)
elevation[:, 7:10]  = np.random.uniform(0.05, 0.9, (GRID, 3))   # river channel
elevation[:, :3]    = np.random.uniform(0.0,  0.5, (GRID, 3))   # coastal strip
elevation           = np.clip(elevation, 0.05, None)

# Land use: 0=Residential  1=Commercial  2=Industrial  3=Green
LU_LABELS = ["Residential", "Commercial", "Industrial", "Green Space"]
LU_COLORS = ["#FFD700",     "#FF6B6B",    "#888888",    "#90EE90"]
landuse = np.full((GRID, GRID), 0, dtype=int)
landuse[:4,  :]     = 1    # northern commercial strip
landuse[16:, :]     = 2    # southern industrial zone
landuse[:, 3:6]     = 3    # green buffer near coast
landuse[7:14, 12:]  = 0    # dense residential inland

# Population density (proxy, people per cell)
BASE_POP = {0: 175, 1: 75, 2: 22, 3: 8}
pop = np.vectorize(lambda lu: BASE_POP[lu])(landuse)
pop = (pop * np.random.uniform(0.6, 1.5, (GRID, GRID))).astype(int)

print("✅ Synthetic city generated")
print(f"   Elevation  : {elevation.min():.1f} m – {elevation.max():.1f} m")
print(f"   Population : {pop.sum():,} people (proxy total)")
print()
for i, label in enumerate(LU_LABELS):
    n = (landuse == i).sum()
    print(f"   {label:14s} : {n:3d} cells  ({100*n/GRID**2:.0f}%)")


In [ ]:
# ── VISUALISE CITY LAYERS ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle(f"Physical Entity Layers — {CITY_NAME}", fontsize=13, fontweight="bold", y=1.01)

# 1. Elevation
im0 = axes[0].imshow(elevation, cmap="terrain", origin="upper", aspect="auto")
axes[0].set_title("🏔️  Elevation (m)")
axes[0].set_xlabel("W  →  E");  axes[0].set_ylabel("N  ↓  S")
plt.colorbar(im0, ax=axes[0], shrink=0.85)
# Mark river channel
for r in range(GRID):
    axes[0].plot([6.6, 9.4], [r, r], color="royalblue", alpha=0.12, lw=0.6)

# 2. Land Use
lu_cmap = ListedColormap(LU_COLORS)
axes[1].imshow(landuse, cmap=lu_cmap, vmin=0, vmax=3, origin="upper", aspect="auto")
axes[1].set_title("🏙️  Land Use")
patches = [mpatches.Patch(color=LU_COLORS[i], label=LU_LABELS[i]) for i in range(4)]
axes[1].legend(handles=patches, loc="lower right", fontsize=7.5, framealpha=0.92)

# 3. Population
im2 = axes[2].imshow(pop, cmap="YlOrRd", origin="upper", aspect="auto")
axes[2].set_title("👥  Population Density (proxy)")
plt.colorbar(im2, ax=axes[2], shrink=0.85, label="people / cell")

plt.tight_layout()
plt.show()

print("\n💡  These three static layers are the 'geometric + semantic' foundation")
print("    of the Digital Twin. In a production system each cell would link to")
print("    real building footprints, census blocks, and infrastructure records.")


---
## Part 3 — Data Acquisition (DT Element: **Da**)

> *"A Digital Twin without live data is just a 3D model."*

The **Data Acquisition** element connects the physical world to the digital replica.

We fetch **real hourly precipitation** from [Open-Meteo](https://open-meteo.com/) —  
a free, open-source weather API, no registration required, global coverage.

| Sensor type | In a real DT | Here |
|-------------|-------------|------|
| Rain gauge network | IoT sensor → MQTT → data broker | Open-Meteo REST API |
| Weather radar | NEXRAD / Doppler → gridded rainfall | Hourly time-series |
| Satellite (GPM) | Near-real-time global precipitation | *(used as extension)* |

> 🚀 **Extension:** Also call `https://flood-api.open-meteo.com/v1/flood` to add  
> **real river discharge** data as a second DT sensor stream.


In [ ]:
# ── FETCH REAL PRECIPITATION — Open-Meteo API ────────────────────────────────
print(f"📡  Fetching real precipitation for {CITY_NAME} ...")

url    = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude":      LAT,
    "longitude":     LON,
    "hourly":        "precipitation",
    "past_days":     7,
    "forecast_days": 1,
    "timezone":      "auto",
}

try:
    r = requests.get(url, params=params, timeout=12)
    r.raise_for_status()
    raw     = r.json()
    times   = pd.to_datetime(raw["hourly"]["time"])
    precip  = np.array(raw["hourly"]["precipitation"], dtype=float)
    print("✅  Live data received from Open-Meteo!")
except Exception as e:
    print(f"⚠️  API unavailable ({e}). Using synthetic fallback.")
    times  = pd.date_range(end=pd.Timestamp.now(), periods=192, freq="h")
    precip = np.abs(np.random.normal(0.4, 1.5, 192))
    precip[precip > 10] = 0.0

df = pd.DataFrame({"time": times, "precip_mm": precip})
df["cumulative_mm"]  = df["precip_mm"].cumsum()
df["rolling_6h_mm"]  = df["precip_mm"].rolling(6, min_periods=1).sum()

TOTAL_RAIN = df["precip_mm"].sum()
PEAK_RAIN  = df["precip_mm"].max()
PEAK_TIME  = df.loc[df["precip_mm"].idxmax(), "time"]

print(f"\n📊  Summary — {CITY_NAME} (7 days + 1-day forecast):")
print(f"    Total rainfall  : {TOTAL_RAIN:.1f} mm")
print(f"    Peak intensity  : {PEAK_RAIN:.1f} mm / hr  at  {PEAK_TIME.strftime('%Y-%m-%d %H:00')}")
print(f"    Rainy hours     : {(df.precip_mm > 0.1).sum()} hrs out of {len(df)}")


In [ ]:
# ── VISUALISE PRECIPITATION ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle(f"📡  Real Precipitation — {CITY_NAME}  (7 days + forecast)",
             fontweight="bold", fontsize=12)

# Hourly bar + rolling average
ax1.bar(df["time"], df["precip_mm"], color="steelblue", alpha=0.65, width=0.04, label="Hourly")
ax1.plot(df["time"], df["rolling_6h_mm"] / 6, color="navy", lw=1.4, label="6-hr rolling avg")
ax1.axhline(PEAK_RAIN, color="crimson", ls="--", lw=0.9, label=f"Peak: {PEAK_RAIN:.1f} mm/hr")
ax1.set_ylabel("mm / hr");  ax1.set_title("Hourly Intensity")
ax1.legend(fontsize=9);     ax1.set_ylim(bottom=0)

# Cumulative
ax2.fill_between(df["time"], df["cumulative_mm"], alpha=0.3, color="navy")
ax2.plot(df["time"], df["cumulative_mm"], color="navy", lw=1.6)
ax2.set_ylabel("Cumulative (mm)");  ax2.set_xlabel("Date / Time")
ax2.set_title("Cumulative Rainfall")
ax2.annotate(
    f"{TOTAL_RAIN:.0f} mm total",
    xy=(df["time"].iloc[-1], TOTAL_RAIN),
    xytext=(-90, -22), textcoords="offset points", fontsize=9, color="navy",
    arrowprops=dict(arrowstyle="->", color="navy"),
)

plt.tight_layout()
plt.show()

print("\n💡  In a production DT this chart would refresh every few minutes")
print("    as sensor data streams in. The DT 'state' is always current.")


---
## Part 4 — Assembling the Digital Twin State

The DT **state** is a snapshot that fuses all data layers at a given moment in time.

Here we combine:
- ✅ Static layers (elevation, land use, population) — from Part 2
- ✅ Live layer (real precipitation) — from Part 3
- ✅ Spatial distribution — rainfall varies spatially across a city

This assembled state feeds directly into the **Simulation** and **Analytics** engines.

> In production systems this fusion step is handled by a **data broker**  
> (e.g. FIWARE Context Broker, Azure Digital Twins, or a custom Kafka pipeline).


In [ ]:
# ── ASSEMBLE DT STATE ─────────────────────────────────────────────────────────
np.random.seed(99)

# Spatially distribute total rainfall across the grid
# (Reality: interpolated from rain gauge network or weather radar)
spatial_var    = np.random.uniform(0.65, 1.38, (GRID, GRID))
rainfall_grid  = TOTAL_RAIN * spatial_var       # mm per cell

# Runoff coefficient: fraction of rain that becomes surface runoff
# Higher impervious surface → more runoff
RUNOFF  = {0: 0.55, 1: 0.85, 2: 0.90, 3: 0.15}
runoff_grid = np.vectorize(lambda lu: RUNOFF[lu])(landuse)

# Infiltration capacity: mm absorbed before runoff begins
INFIL   = {0: 22, 1: 8, 2: 5, 3: 65}
infil_grid  = np.vectorize(lambda lu: INFIL[lu])(landuse)

# DT State dictionary (what a real system would log)
dt_state = {
    "city":          CITY_NAME,
    "timestamp":     pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "total_rain_mm": round(TOTAL_RAIN, 1),
    "peak_mm_hr":    round(PEAK_RAIN, 1),
    "grid_cells":    GRID * GRID,
    "active_layers": ["elevation", "landuse", "population", "precipitation", "runoff"],
}

print("🗂️   Digital Twin State assembled:")
for k, v in dt_state.items():
    print(f"    {k:18s}: {v}")

print(f"\n    Rainfall grid  : {rainfall_grid.min():.1f} – {rainfall_grid.max():.1f} mm (spatial range)")
print(f"    Surface runoff : {(rainfall_grid * runoff_grid).mean():.1f} mm (grid mean)")


---
## Part 5 — Flood Simulation (DT Element: **Si**)

The **Simulation** element transforms raw DT state data into **predictive spatial insight**.

We implement a **Modified Bathtub Model** — conceptually transparent and fast to compute:

```
surface_runoff = rainfall × runoff_coefficient
excess_water   = max(0,  surface_runoff  −  infiltration_capacity)
flood_index    = excess_water ÷ (elevation + buffer)
```

Low elevation + high impervious surface + heavy rainfall = high flood index.

---

> **Honest limitations:** The bathtub model ignores flow routing, drainage networks,  
> storm sewer capacity, and temporal dynamics. Production DTs use physics-based solvers  
> (**HEC-RAS**, **ADCIRC**, **MIKE FLOOD**) that require hours of compute time.  
> **ML surrogate models** replicate their outputs in milliseconds — that's where  
> *AI-enabled DTs* (our element **An**) close the speed-accuracy gap.

> 🚀 **Extension:** Add a `DRAINAGE_CAPACITY` parameter per land use and re-run the model.  
> Observe how upgrading stormwater infrastructure changes the flood risk map.


In [ ]:
# ── FLOOD SUSCEPTIBILITY MODEL ────────────────────────────────────────────────

# Step 1: Surface runoff (mm) reaching the ground surface
surface_runoff = rainfall_grid * runoff_grid

# Step 2: Net excess after soil infiltration
excess_water   = np.maximum(0, surface_runoff - infil_grid)

# Step 3: Flood susceptibility index
#   Water ponds in low-lying areas → divide excess by elevation
elev_buffer    = elevation + 0.35          # prevent division by near-zero
flood_idx      = excess_water / elev_buffer

# Step 4: Threshold + normalise to intuitive 0–2.5 m depth range
threshold      = np.percentile(flood_idx, 38)
flood_raw      = np.maximum(0, flood_idx - threshold)
flood_m        = (flood_raw / flood_raw.max() * 2.5) if flood_raw.max() > 0 else flood_raw

# ── FLOOD CLASSIFICATION ──────────────────────────────────────────────────────
def classify(d):
    if   d < 0.05: return 0   # No flood
    elif d < 0.50: return 1   # Minor     (< ankle)
    elif d < 1.00: return 2   # Moderate  (knee-deep)
    elif d < 1.80: return 3   # Major     (chest-deep)
    else:          return 4   # Extreme   (> 1.8 m)

flood_class  = np.vectorize(classify)(flood_m)
CLS_LABELS   = ["No Flood", "Minor < 0.5m", "Moderate 0.5–1m", "Major 1–1.8m", "Extreme > 1.8m"]
CLS_COLORS   = ["#e8f4f8",  "#ffffb2",       "#fd8d3c",          "#d7191c",       "#6d0000"]

flooded      = (flood_class > 0).sum()
affected_pop = pop[flood_class > 0].sum()
critical     = (flood_class >= 3).sum()

print("🌊  Simulation complete!")
print(f"    Flooded cells      : {flooded} / {GRID**2}  ({100*flooded/GRID**2:.0f}%)")
print(f"    Affected population: {affected_pop:,} (proxy)")
print(f"    Critical zones     : {critical} cells at Major or Extreme depth")
print(f"    Max depth index    : {flood_m.max():.2f} m\n")
for cid, label in enumerate(CLS_LABELS):
    n   = (flood_class == cid).sum()
    bar = "█" * n
    print(f"    {label:22s}: {n:3d} cells  {bar}")


In [ ]:
# ── VISUALISE SIMULATION RESULTS ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"🌊  Flood Simulation Results — {CITY_NAME}", fontsize=13, fontweight="bold")

# Panel 1: Elevation reference
im0 = axes[0].imshow(elevation, cmap="terrain", origin="upper", aspect="auto")
axes[0].set_title("Terrain Elevation (m)")
axes[0].set_xlabel("W → E"); axes[0].set_ylabel("N → S")
plt.colorbar(im0, ax=axes[0], shrink=0.85)

# Panel 2: Flood depth index
flood_cmap = LinearSegmentedColormap.from_list(
    "flood", ["#e8f4f8","#ffffb2","#fd8d3c","#d7191c","#6d0000"])
im1 = axes[1].imshow(flood_m, cmap=flood_cmap, vmin=0, vmax=2.5, origin="upper", aspect="auto")
axes[1].set_title("Flood Depth Index (m, conceptual)")
plt.colorbar(im1, ax=axes[1], shrink=0.85, label="Depth (m)")

# Panel 3: Class map
cls_cmap = ListedColormap(CLS_COLORS)
axes[2].imshow(flood_class, cmap=cls_cmap, vmin=0, vmax=4, origin="upper", aspect="auto")
axes[2].set_title("Flood Classification")
patches = [mpatches.Patch(color=CLS_COLORS[i], label=CLS_LABELS[i]) for i in range(5)]
axes[2].legend(handles=patches, loc="lower right", fontsize=7, framealpha=0.92)

plt.tight_layout()
plt.show()

print("\n💡  Compare the elevation map (left) with the flood classification (right).")
print("    Notice: Commercial / Industrial zones at LOW elevation are hit hardest.")
print("    Green-space buffers reduce flooding even at similar elevations.")


---
## Part 6 — Interactive Visualization (DT Element: **Vi**)

A key value of a Digital Twin is making simulation outputs **accessible to operators  
and decision-makers** — not just data scientists.

We place the synthetic flood grid on the **real city coordinates** using Folium.  
Each circle marker = one grid cell. Click any marker to see cell-level details.

> 🚀 **Extension:** Replace `CircleMarker` with `folium.plugins.HeatMap` for a  
> continuous density surface, or export cells as GeoJSON and load into QGIS.


In [ ]:
# ── INTERACTIVE FOLIUM MAP ─────────────────────────────────────────────────────
CELL_DEG = 0.008     # ~700m per cell at mid-latitudes (approximate)
CLS_FOLIUM = ["lightblue", "beige", "orange", "red", "darkred"]

m = folium.Map(location=[LAT, LON], zoom_start=12, tiles="CartoDB positron")

# Legend HTML
legend = (
    '<div style="position:fixed;bottom:28px;left:28px;z-index:1000;background:white;'
    'padding:12px 14px;border-radius:8px;border:1px solid #ccc;font-size:12px;'
    'box-shadow:2px 2px 6px rgba(0,0,0,0.15);">'
    '<b>&#127754; Flood Depth Class</b><br>'
    '<span style="color:#9ecae1">&#9679;</span> No Flood<br>'
    '<span style="color:#fed976">&#9679;</span> Minor (&lt; 0.5 m)<br>'
    '<span style="color:#fd8d3c">&#9679;</span> Moderate (0.5 - 1 m)<br>'
    '<span style="color:#d7191c">&#9679;</span> Major (1 - 1.8 m)<br>'
    '<span style="color:#6d0000">&#9679;</span> Extreme (&gt; 1.8 m)'
    '</div>'
)
m.get_root().html.add_child(folium.Element(legend))

# Grid cells → map markers
for i in range(GRID):
    for j in range(GRID):
        cls   = int(flood_class[i, j])
        if cls == 0:
            continue     # skip unflooded for clarity
        cell_lat = LAT + (i - GRID // 2) * CELL_DEG
        cell_lon = LON + (j - GRID // 2) * CELL_DEG
        popup_html = (
            f"<b>{CITY_NAME} — Cell ({i},{j})</b><br>"
            f"Flood Class : <b>{CLS_LABELS[cls]}</b><br>"
            f"Depth Index : {flood_m[i,j]:.2f} m<br>"
            f"Elevation   : {elevation[i,j]:.2f} m<br>"
            f"Land Use    : {LU_LABELS[landuse[i,j]]}<br>"
            f"Population  : ~{pop[i,j]} people"
        )
        folium.CircleMarker(
            location=[cell_lat, cell_lon],
            radius=8, color="white", weight=0.5,
            fill=True, fill_color=CLS_FOLIUM[cls], fill_opacity=0.78,
            popup=folium.Popup(popup_html, max_width=230),
            tooltip=f"{CLS_LABELS[cls]}  |  {LU_LABELS[landuse[i,j]]}",
        ).add_to(m)

# City centre pin
folium.Marker(
    location=[LAT, LON],
    popup=f"<b>{CITY_NAME} City Centre</b><br>Total rain: {TOTAL_RAIN:.0f} mm",
    icon=folium.Icon(color="blue", icon="home"),
    tooltip="City centre",
).add_to(m)

display(m)
print("\n👆  Click any marker for cell details. Red/dark-red = highest priority zones.")


---
## Part 7 — Analytics & Decision Support (DT Element: **An**)

The **Analytics** element is where the DT transforms simulation outputs into  
**actionable intelligence** for operators and emergency managers.

A well-designed DT doesn't just show *what is happening* —  
it tells decision-makers *what to do, when, and at what update rate*.

We implement two analytical outputs:
1. **Risk Score** — per-cell composite of flood depth × population × vulnerability
2. **Tiered Recommendations** — structured decision support table by risk tier


In [ ]:
# ── RISK SCORING ──────────────────────────────────────────────────────────────
# Risk = Flood Depth × Population Density × Land-Use Vulnerability
VULN = {0: 1.25, 1: 0.75, 2: 0.55, 3: 0.25}   # residential most vulnerable
vuln_grid  = np.vectorize(lambda lu: VULN[lu])(landuse)
risk_raw   = flood_m * pop * vuln_grid

# Normalise 0–100
risk_score = (risk_raw / risk_raw.max() * 100) if risk_raw.max() > 0 else risk_raw

def tier(s):
    if   s >= 75: return "CRITICAL"
    elif s >= 50: return "HIGH"
    elif s >= 20: return "MODERATE"
    elif s >   0: return "LOW"
    return "NONE"

risk_tier  = np.vectorize(tier)(risk_score)

# ── VISUALISE RISK ─────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"📊  Risk Analytics — {CITY_NAME}", fontsize=13, fontweight="bold")

risk_cmap = LinearSegmentedColormap.from_list(
    "risk", ["#f7f7f7","#fee08b","#fc8d59","#d73027","#67001f"])
im0 = ax1.imshow(risk_score, cmap=risk_cmap, vmin=0, vmax=100, origin="upper", aspect="auto")
ax1.set_title("Composite Risk Score (0–100)")
plt.colorbar(im0, ax=ax1, shrink=0.85)

TIER_COL = {"CRITICAL":"#67001f","HIGH":"#d73027","MODERATE":"#fc8d59","LOW":"#fee08b","NONE":"#f7f7f7"}
for t, c in TIER_COL.items():
    mask = risk_tier == t
    if mask.any():
        rows, cols = np.where(mask)
        ax2.scatter(cols, GRID - rows, c=c, s=80,
                    label=f"{t}: {mask.sum()} cells", zorder=3)

ax2.set_xlim(-1, GRID); ax2.set_ylim(-1, GRID + 1)
ax2.set_title("Risk Tier Map")
ax2.legend(loc="upper left", fontsize=8, framealpha=0.93)
ax2.set_aspect("equal")
plt.tight_layout()
plt.show()

print("\n📊  Risk Tier Summary:")
for t in ["CRITICAL","HIGH","MODERATE","LOW","NONE"]:
    n   = (risk_tier == t).sum()
    pop_at_risk = pop[risk_tier == t].sum()
    bar = "█" * n
    print(f"    {t:10s}: {n:3d} cells  ~{pop_at_risk:,} people  {bar}")


In [ ]:
# ── RECOMMENDATION ENGINE ────────────────────────────────────────────────────
REC = {
    "CRITICAL": {
        "action":   "🚨  IMMEDIATE evacuation — deploy emergency response teams NOW",
        "cadence":  "Real-time (every 1–5 min)",
        "watch":    "Stream gauges, road passability, shelter occupancy",
        "owner":    "Emergency Management Director",
    },
    "HIGH": {
        "action":   "⚠️   Pre-position resources — open shelters, alert all residents",
        "cadence":  "Every 15 minutes",
        "watch":    "Rain radar, upstream gauges, hospital surge capacity",
        "owner":    "County Emergency Coordinator",
    },
    "MODERATE": {
        "action":   "📢  Issue advisory — prepare evacuation routes and pre-stage equipment",
        "cadence":  "Hourly",
        "watch":    "Precipitation forecast, drainage inlet flow sensors",
        "owner":    "City Operations Center",
    },
    "LOW": {
        "action":   "ℹ️   Notify community — no immediate action, continue monitoring",
        "cadence":  "Every 6 hours",
        "watch":    "24-hr rainfall forecast updates",
        "owner":    "Duty Officer",
    },
}

LINE = "─" * 74
print("=" * 74)
print(f"  DIGITAL TWIN DECISION SUPPORT REPORT  —  {CITY_NAME.upper()}")
print(f"  Rainfall: {TOTAL_RAIN:.1f} mm  |  Peak: {PEAK_RAIN:.1f} mm/hr  |"
      f"  Exposed population: ~{pop[flood_class > 0].sum():,}")
print("=" * 74)

for t in ["CRITICAL","HIGH","MODERATE","LOW"]:
    n   = (risk_tier == t).sum()
    if n == 0:
        continue
    p   = pop[risk_tier == t].sum()
    rec = REC[t]
    print(f"\n{LINE}")
    print(f"  [{t}]   {n} zones   |   ~{p:,} people at risk")
    print(f"  Action      : {rec['action']}")
    print(f"  DT Cadence  : {rec['cadence']}")
    print(f"  Watch       : {rec['watch']}")
    print(f"  Owner       : {rec['owner']}")

print(f"\n{LINE}")
print("  MODEL LIMITATIONS (always document these in real DTs):")
print("  • Bathtub model — no flow routing or drainage infrastructure modelled")
print("  • No time-stepping — represents accumulated 7-day rainfall, not peak event")
print("  • Synthetic terrain — must be replaced with real DEM for production use")
print("  • Population proxy — use census block data for real exposure estimates")
print("=" * 74)


---
## Part 8 — Reflection & Next Steps

### 🎯 What We Built

| Component | What We Did | Real DT Equivalent |
|-----------|-------------|-------------------|
| Physical entity | 20×20 synthetic city grid | GIS + BIM + cadastral data |
| **Da** Data Acquisition | Open-Meteo real precipitation API | IoT gauge network + weather radar |
| DT State fusion | Spatially distributed rainfall + terrain | Real-time sensor fusion engine |
| **Si** Simulation | Modified bathtub flood model | HEC-RAS / ADCIRC / MIKE FLOOD |
| **Vi** Visualization | Folium interactive map | Cesium / ArcGIS / Bentley iTwin |
| **An** Analytics | Risk scoring + tiered recommendations | FEMA HAZUS / custom DSS |

---

### 🔢 DT Periodic Table — What We Touched

| Element | In this notebook |
|---------|----------------|
| **Da** — Data Acquisition | Real API call → 192 hours of hourly precipitation |
| **Si** — Simulation | Bathtub model → flood depth index per cell |
| **An** — Analytics | Risk score → structured decision support table |

---

### 🚀 Extension Challenges

1. **Easy:** Change `SELECTED` in Cell 2 to another city and compare flood profiles
2. **Easy:** Adjust `GRID = 30` for a finer resolution grid and re-run all cells
3. **Medium:** Call the [Open-Meteo Flood API](https://flood-api.open-meteo.com/v1/flood)
   and add river discharge as a second data stream in the DT State
4. **Medium:** Add a `DRAINAGE_CAPACITY` parameter per land use and see how upgrading  
   infrastructure changes the risk map
5. **Hard:** Replace the synthetic elevation with a real DEM from  
   [OpenTopography](https://opentopography.org) using their API
6. **Hard:** Add time-stepping — show how flood depth evolves hour-by-hour  
   using `matplotlib.animation` or an `ipywidgets` slider

---

### 📚 Further Resources

| Resource | Link |
|----------|------|
| Open-Meteo API docs | https://open-meteo.com/en/docs |
| Digital Twin Consortium | https://digitaltwinconsortium.org |
| Cesium 3D DT platform | https://cesium.com/platform |
| ADCIRC storm surge model | https://adcirc.org |
| OGC Disaster Pilot reports | https://www.ogc.org/initiatives/disaster-pilot |
| FIWARE context broker | https://fiware.org |

---

> *A Digital Twin is not a product — it is a living data system that evolves alongside its physical counterpart.*  
> *Today you built the skeleton. The rest is real data, compute, and cross-sector collaboration.*
